# Sprint 2 Colab Pipeline

This notebook supports exactly one Sprint 2 execution path:

1. Mount Google Drive at `/content/drive`.
2. Read translations only from:
   - `/content/drive/MyDrive/text-to-sign-production/raw/how2sign/translations/how2sign_realigned_train.csv`
   - `/content/drive/MyDrive/text-to-sign-production/raw/how2sign/translations/how2sign_realigned_val.csv`
   - `/content/drive/MyDrive/text-to-sign-production/raw/how2sign/translations/how2sign_realigned_test.csv`
3. Read keypoints only from `.tar.zst` archives at:
   - `/content/drive/MyDrive/text-to-sign-production/raw/how2sign/archives/train_2D_keypoints.tar.zst`
   - `/content/drive/MyDrive/text-to-sign-production/raw/how2sign/archives/val_2D_keypoints.tar.zst`
   - `/content/drive/MyDrive/text-to-sign-production/raw/how2sign/archives/test_2D_keypoints.tar.zst`
4. Copy archives into `/content/how2sign_downloads`, extract there, and stage the canonical repo raw layout.
5. Run the existing Sprint 2 scripts.
6. Package outputs and copy the archives only to `/content/drive/MyDrive/text-to-sign-production/artifacts/sprint2/processed-v1/`.

The notebook is orchestration-only. Long-running copy, extract, archive, and publish logic lives in repository scripts and modules.


## 1. Environment And Repository Setup

This section installs the minimal Colab prerequisites, clones the repository, installs Python dependencies, and defines a small command runner.


In [ ]:
from __future__ import annotations

import os
import shlex
import shutil
import subprocess
import sys
from pathlib import Path

print(f"Python: {sys.version.split()[0]}")

if shutil.which("zstd") is None:
    subprocess.run(["apt-get", "update"], check=True)
    subprocess.run(["apt-get", "install", "-y", "zstd"], check=True)
else:
    print("zstd is already available.")

REPO_URL = "https://github.com/0xmillennium/text-to-sign-production.git"
REPO_REF = "master"
WORKTREE_ROOT = Path("/content/text-to-sign-production")
RUNTIME_ROOT = WORKTREE_ROOT.parent

RUNTIME_ROOT.mkdir(parents=True, exist_ok=True)
try:
    current_cwd = Path.cwd()
except FileNotFoundError:
    current_cwd = None

if current_cwd is None or current_cwd == WORKTREE_ROOT or WORKTREE_ROOT in current_cwd.parents:
    os.chdir(RUNTIME_ROOT)

if WORKTREE_ROOT.exists():
    shutil.rmtree(WORKTREE_ROOT)

subprocess.run(["git", "clone", REPO_URL, str(WORKTREE_ROOT)], check=True)
subprocess.run(["git", "checkout", REPO_REF], cwd=WORKTREE_ROOT, check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"], cwd=WORKTREE_ROOT, check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements-colab.txt"], cwd=WORKTREE_ROOT, check=True)
os.chdir(WORKTREE_ROOT)

current_revision = subprocess.run(
    ["git", "rev-parse", "HEAD"],
    cwd=WORKTREE_ROOT,
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
print(f"Repository ready at {WORKTREE_ROOT}")
print(f"Checked out revision: {current_revision}")

def run_command(command: list[str]) -> None:
    print(f"$ {shlex.join(command)}")
    subprocess.run(command, cwd=WORKTREE_ROOT, check=True)


## 2. Mount Google Drive

Edit only `PIPELINE_SPLITS` if you want a subset such as `["val"]` or `["test"]`. All input and output paths are fixed by the repository.


In [ ]:
from pathlib import Path

from google.colab import drive

PIPELINE_SPLITS = ["train", "val", "test"]

drive.mount("/content/drive", force_remount=False)
print(f"Selected splits: {PIPELINE_SPLITS}")
print("Fixed input root: /content/drive/MyDrive/text-to-sign-production/raw/how2sign")
print("Fixed local archive workspace: /content/how2sign_downloads")
print("Fixed output root: /content/drive/MyDrive/text-to-sign-production/artifacts/sprint2/processed-v1/")


## 3. Stage Fixed Colab Inputs

This calls the repository staging script, which copies the fixed Drive archives into local runtime storage, extracts only `.tar.zst`, stages the canonical raw layout, and prints progress while it works.


In [ ]:
run_command([sys.executable, "scripts/stage_colab_inputs.py", "--splits", *PIPELINE_SPLITS])
print("Canonical raw layout staged.")


## 4. Run Sprint 2 Pipeline Scripts

This notebook stays orchestration-only by invoking the existing repository scripts directly.


In [ ]:
commands = [
    [sys.executable, "scripts/prepare_raw.py", "--splits", *PIPELINE_SPLITS],
    [sys.executable, "scripts/normalize_keypoints.py", "--splits", *PIPELINE_SPLITS],
    [sys.executable, "scripts/filter_samples.py", "--config", "configs/data/filter-v1.yaml", "--splits", *PIPELINE_SPLITS],
    [sys.executable, "scripts/export_training_manifest.py", "--splits", *PIPELINE_SPLITS],
]

for command in commands:
    run_command(command)

print("Sprint 2 pipeline completed.")


## 5. Package And Publish Outputs

This calls the repository publish script, which creates `.tar.zst` archives with progress and copies them to the fixed Drive artifact destination with progress.


In [ ]:
run_command([sys.executable, "scripts/publish_colab_outputs.py", "--splits", *PIPELINE_SPLITS])
print("Published Sprint 2 archives to /content/drive/MyDrive/text-to-sign-production/artifacts/sprint2/processed-v1/")
